# Gabor filter intuition
Step-by-step construction with sliders. Each panel shows one ingredient.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, ToggleButton, ToggleButtons
import math

# --- coordinate grid (fixed) ---
SIZE = 64
half = SIZE // 2
xs = np.arange(SIZE, dtype=np.float32) - half
ys = np.arange(SIZE, dtype=np.float32) - half
yy, xx = np.meshgrid(ys, xs, indexing='ij')   # xx rightward, yy downward

def make_gabor(theta, lam, sigma, gamma, psi,
               zero_xr=False, zero_yr=False):
    """
    theta  : orientation [deg]  0=horizontal bars, 90=vertical bars
    lam    : wavelength [px]    period of the cosine carrier
    sigma  : envelope width [px]
    gamma  : aspect ratio       >1 elongated along bars, <1 along oscillation
    psi    : carrier phase [deg]
    zero_xr: replace x_r with zeros (kills carrier + narrows envelope)
    zero_yr: replace y_r with zeros (kills elongation)
    """
    t = math.radians(theta)
    p = math.radians(psi)

    # Rotated coordinates
    # x_r : along oscillation direction  (perpendicular to bars)
    # y_r : along bar direction           (parallel to bars)
    x_r = -xx * math.sin(t) + yy * math.cos(t)
    y_r =  xx * math.cos(t) + yy * math.sin(t)

    if zero_xr:
        x_r = np.zeros_like(x_r)
    if zero_yr:
        y_r = np.zeros_like(y_r)

    envelope = np.exp(-(x_r**2 + (gamma * y_r)**2) / (2.0 * sigma**2))
    carrier  = np.cos(2.0 * math.pi * x_r / lam + p)
    gabor    = envelope * carrier

    return x_r, y_r, envelope, carrier, gabor


def plot_gabor(theta=0.0, lam=12.0, sigma=8.0, gamma=1.0, psi=0.0,
               zero_xr=False, zero_yr=False):
    x_r, y_r, envelope, carrier, gabor = make_gabor(
        theta, lam, sigma, gamma, psi, zero_xr, zero_yr
    )

    fig, axes = plt.subplots(1, 5, figsize=(18, 4))
    cmap_signed = 'RdBu_r'
    cmap_pos    = 'viridis'

    def show(ax, data, title, cmap, vabs=None):
        vmax = vabs if vabs else np.abs(data).max() + 1e-6
        vmin = -vmax if cmap == cmap_signed else 0
        im = ax.imshow(data, cmap=cmap, vmin=vmin, vmax=vmax,
                       origin='upper', extent=[-half, half, half, -half])
        ax.set_title(title, fontsize=11)
        ax.axhline(0, color='k', lw=0.5, alpha=0.4)
        ax.axvline(0, color='k', lw=0.5, alpha=0.4)
        plt.colorbar(im, ax=ax, fraction=0.046)

    show(axes[0], x_r,      'x_r  (oscillation axis)\n← zeroed →' if zero_xr else 'x_r  (oscillation axis)',
         cmap_signed)
    show(axes[1], y_r,      'y_r  (bar axis)\n← zeroed →' if zero_yr else 'y_r  (bar axis)',
         cmap_signed)
    show(axes[2], envelope, 'envelope\nexp(−(x_r² + (γ·y_r)²) / 2σ²)',
         cmap_pos, vabs=1.0)
    show(axes[3], carrier,  'carrier\ncos(2π·x_r/λ + ψ)',
         cmap_signed, vabs=1.0)
    show(axes[4], gabor,    'Gabor = envelope × carrier',
         cmap_signed)

    # Overlay bar direction arrow on Gabor panel
    t = math.radians(theta)
    axes[4].annotate('', xy=(20*math.cos(t), 20*math.sin(t)),
                     xytext=(0, 0),
                     arrowprops=dict(arrowstyle='->', color='lime', lw=2))
    axes[4].set_title(f'Gabor  θ={theta:.0f}° λ={lam:.1f} σ={sigma:.1f} γ={gamma:.2f} ψ={psi:.0f}°',
                      fontsize=10)

    plt.tight_layout()
    plt.show()


interact(
    plot_gabor,
    theta   = FloatSlider(value=0,   min=0,   max=180, step=5,   description='θ [deg]',
                          continuous_update=True, style={'description_width': '80px'}),
    lam     = FloatSlider(value=12,  min=2,   max=40,  step=0.5, description='λ wavelength',
                          continuous_update=True, style={'description_width': '80px'}),
    sigma   = FloatSlider(value=8,   min=1,   max=30,  step=0.5, description='σ envelope',
                          continuous_update=True, style={'description_width': '80px'}),
    gamma   = FloatSlider(value=1.0, min=0.1, max=4.0, step=0.05,description='γ aspect',
                          continuous_update=True, style={'description_width': '80px'}),
    psi     = FloatSlider(value=0,   min=-180,max=180, step=5,   description='ψ phase',
                          continuous_update=True, style={'description_width': '80px'}),
    zero_xr = ToggleButton(value=False, description='zero x_r',
                           button_style='warning', tooltip='Set x_r=0 everywhere'),
    zero_yr = ToggleButton(value=False, description='zero y_r',
                           button_style='info',    tooltip='Set y_r=0 everywhere'),
);

interactive(children=(FloatSlider(value=0.0, description='θ [deg]', max=180.0, step=5.0, style=SliderStyle(des…

## What x_r and y_r actually are

They are the **original (xx, yy) pixel grid rotated by θ**:

```
x_r = −xx·sin(θ) + yy·cos(θ)    ← perpendicular to the bars
y_r =  xx·cos(θ) + yy·sin(θ)    ← parallel to the bars
```

Think of it as laying a new ruler across the image at angle θ:

| coordinate | measures distance... | used in...       | effect if zeroed |
|------------|----------------------|------------------|------------------|
| **x_r**    | across the bars      | carrier + envelope | carrier becomes `cos(ψ)` = flat constant; envelope shrinks to a line along y_r |
| **y_r**    | along the bars       | envelope only    | envelope becomes isotropic (γ has no effect) |

### Why two coordinates instead of one?
A Gabor is **anisotropic**: it should be wide along the bars and narrow across them.  
The `gamma` parameter stretches the Gaussian *only* along `y_r`, controlling that elongation independently of `sigma`.

In [2]:
# --- Cross-section plots: see the 1-D profile along each axis ---

def plot_crosssections(theta=0.0, lam=12.0, sigma=8.0, gamma=1.5, psi=0.0):
    x_r, y_r, envelope, carrier, gabor = make_gabor(theta, lam, sigma, gamma, psi)

    # Sample 1-D slices through the centre of x_r and y_r axes
    # For theta=0: x_r varies along columns (yy direction), y_r along rows (xx direction)
    mid = SIZE // 2

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Profile along x_r axis (take the column where y_r ≈ 0)
    ax = axes[0]
    ax.plot(xs, gabor[mid, :],    label='Gabor row', lw=2)
    ax.plot(xs, envelope[mid, :], label='envelope',  lw=2, linestyle='--')
    ax.plot(xs, carrier[mid, :],  label='carrier',   lw=1, alpha=0.6)
    ax.axhline(0, color='k', lw=0.5)
    ax.set_title('Profile along x axis (centre row)')
    ax.set_xlabel('pixel position')
    ax.legend()

    ax = axes[1]
    ax.plot(ys, gabor[:, mid],    label='Gabor col', lw=2)
    ax.plot(ys, envelope[:, mid], label='envelope',  lw=2, linestyle='--')
    ax.plot(ys, carrier[:, mid],  label='carrier',   lw=1, alpha=0.6)
    ax.axhline(0, color='k', lw=0.5)
    ax.set_title('Profile along y axis (centre column)')
    ax.set_xlabel('pixel position')
    ax.legend()

    plt.suptitle(f'θ={theta:.0f}°  λ={lam:.1f}  σ={sigma:.1f}  γ={gamma:.2f}  ψ={psi:.0f}°',
                 fontsize=12)
    plt.tight_layout()
    plt.show()

interact(
    plot_crosssections,
    theta = FloatSlider(value=0,   min=0,    max=180, step=5,   description='θ [deg]'),
    lam   = FloatSlider(value=12,  min=2,    max=40,  step=0.5, description='λ wavelength'),
    sigma = FloatSlider(value=8,   min=1,    max=30,  step=0.5, description='σ envelope'),
    gamma = FloatSlider(value=1.5, min=0.1,  max=4.0, step=0.05,description='γ aspect'),
    psi   = FloatSlider(value=0,   min=-180, max=180, step=5,   description='ψ phase'),
);

interactive(children=(FloatSlider(value=0.0, description='θ [deg]', max=180.0, step=5.0), FloatSlider(value=12…